# Gradient-Based Similarity Analysis Demo

This notebook demonstrates how to use gradient-based similarity measures to compare neural networks.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add parent directory to path
sys.path.append('..')

from src.analysis.gradient_similarity import GradientSimilarityAnalyzer
from src.models.torch_mlp import MLP, generate_torus_data
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline
plt.style.use('seaborn-v0_8')

## 1. Setup and Data Preparation

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Generate synthetic data
X, y = generate_torus_data(n=1000, big_radius=3, small_radius=1, solid=False)
X = X.to(device)
y = y.to(device).float()

# Split data
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Create data loaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")

## 2. Create Models with Different Initializations

In [ ]:
# Create multiple models with different architectures and initializations
models = {
    'shallow': MLP(input_dim=3, num_hidden_layers=2, hidden_dim=32, output_dim=1),
    'deep': MLP(input_dim=3, num_hidden_layers=4, hidden_dim=32, output_dim=1),
    'wide': MLP(input_dim=3, num_hidden_layers=2, hidden_dim=64, output_dim=1),
    'narrow': MLP(input_dim=3, num_hidden_layers=2, hidden_dim=16, output_dim=1),
}

# Move models to device
for name, model in models.items():
    model.to(device)
    print(f"{name}: {sum(p.numel() for p in model.parameters())} parameters")

## 3. Initialize Gradient Similarity Analyzer

In [ ]:
# Initialize analyzer
analyzer = GradientSimilarityAnalyzer(device=device)

# Loss function
loss_fn = nn.BCELoss()

## 4. Gradient Flow Analysis

In [ ]:
# Track gradient flows for each model
gradient_flows = {}

for name, model in models.items():
    print(f"\nTracking gradient flow for {name} model...")
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    flow = analyzer.track_gradient_flow(
        model, loss_fn, train_loader, optimizer, num_steps=30
    )
    gradient_flows[name] = flow
    
    print(f"  Final loss: {flow[-1].loss:.4f}")
    print(f"  Final gradient norm: {flow[-1].gradients.norm().item():.4f}")

In [ ]:
# Visualize gradient flows
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, flow) in enumerate(gradient_flows.items()):
    steps = [s.step for s in flow]
    losses = [s.loss for s in flow]
    grad_norms = [s.gradients.norm().item() for s in flow]
    
    ax = axes[idx]
    ax2 = ax.twinx()
    
    line1 = ax.plot(steps, losses, 'b-', label='Loss')
    line2 = ax2.plot(steps, grad_norms, 'r-', label='Gradient Norm')
    
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss', color='b')
    ax2.set_ylabel('Gradient Norm', color='r')
    ax.set_title(f'{name.capitalize()} Model')
    
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax.legend(lines, labels, loc='upper right')

plt.tight_layout()
plt.show()

## 5. Loss Landscape Analysis

In [ ]:
# Analyze loss landscape for each model
landscapes = {}

for name, model in models.items():
    print(f"\nAnalyzing loss landscape for {name} model...")
    
    landscape = analyzer.analyze_loss_landscape(
        model, loss_fn, train_loader,
        resolution=25,  # Lower resolution for faster computation
        epsilon=0.15
    )
    landscapes[name] = landscape
    
    print(f"  Roughness: {landscape['roughness']:.4f}")
    print(f"  Convexity: {landscape['convexity']:.4f}")
    print(f"  Basin volume: {landscape['basin_volume']:.4f}")

In [ ]:
# Visualize loss landscapes
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, landscape) in enumerate(landscapes.items()):
    ax = axes[idx]
    
    surface = landscape['loss_surface']
    im = ax.imshow(surface, cmap='viridis', origin='lower')
    ax.set_title(f'{name.capitalize()} Model Loss Landscape')
    ax.set_xlabel('Direction 1')
    ax.set_ylabel('Direction 2')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Loss')
    
    # Add contour lines
    ax.contour(surface, levels=10, colors='white', alpha=0.5, linewidths=0.5)

plt.tight_layout()
plt.show()

## 6. Hessian Analysis

In [ ]:
# Analyze curvature properties
curvatures = {}

for name, model in models.items():
    print(f"\nAnalyzing curvature for {name} model...")
    
    curvature = analyzer.analyze_curvature(
        model, loss_fn, train_loader,
        num_directions=15
    )
    curvatures[name] = curvature
    
    print(f"  Mean curvature: {curvature['mean_curvature']:.4f}")
    print(f"  Negative curvature ratio: {curvature['negative_curvature_ratio']:.4f}")

In [ ]:
# Visualize curvature distributions
plt.figure(figsize=(10, 6))

for name, curvature in curvatures.items():
    plt.hist(curvature['curvature_distribution'], bins=20, alpha=0.6, label=name)

plt.axvline(x=0, color='red', linestyle='--', label='Zero curvature')
plt.xlabel('Directional Curvature')
plt.ylabel('Count')
plt.title('Curvature Distribution Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Neural Tangent Kernel Analysis

In [ ]:
# Sample data for NTK computation
x_sample = X_train[:50]

# Compute NTK for each model
ntk_matrices = {}

for name, model in models.items():
    print(f"\nComputing NTK for {name} model...")
    
    with torch.no_grad():
        K = analyzer.compute_ntk(model, x_sample, x_sample)
        ntk_matrices[name] = K.cpu().numpy()
    
    print(f"  NTK shape: {K.shape}")
    print(f"  NTK trace: {K.trace().item():.2f}")

In [ ]:
# Visualize NTK matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, K) in enumerate(ntk_matrices.items()):
    ax = axes[idx]
    
    # Normalize for visualization
    K_norm = K / (np.abs(K).max() + 1e-8)
    
    im = ax.imshow(K_norm, cmap='RdBu', vmin=-1, vmax=1)
    ax.set_title(f'{name.capitalize()} Model NTK')
    ax.set_xlabel('Sample i')
    ax.set_ylabel('Sample j')
    
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## 8. Pairwise Model Comparison

In [ ]:
# Compute pairwise similarities
model_names = list(models.keys())
n_models = len(model_names)

# Initialize similarity matrices
flow_similarities = np.eye(n_models)
landscape_similarities = np.eye(n_models)
hessian_similarities = np.eye(n_models)
ntk_similarities = np.eye(n_models)

# Compute all pairwise similarities
for i in range(n_models):
    for j in range(i+1, n_models):
        name1, name2 = model_names[i], model_names[j]
        print(f"\nComparing {name1} vs {name2}...")
        
        # Gradient flow similarity
        flow_sim = analyzer.compute_gradient_flow_similarity(
            gradient_flows[name1], gradient_flows[name2], method='velocity'
        )
        flow_similarities[i, j] = flow_similarities[j, i] = flow_sim
        
        # Loss landscape similarity
        landscape_sim = analyzer.compare_loss_landscapes(
            landscapes[name1], landscapes[name2]
        )
        landscape_similarities[i, j] = landscape_similarities[j, i] = landscape_sim['surface_correlation']
        
        # Hessian similarity
        hessian_sim = analyzer.compute_hessian_similarity(
            models[name1], models[name2], loss_fn, train_loader,
            method='eigenvalue', top_k=20
        )
        hessian_similarities[i, j] = hessian_similarities[j, i] = hessian_sim
        
        # NTK similarity
        ntk_sim = analyzer.compare_ntk_similarity(
            models[name1], models[name2], train_loader,
            num_samples=50
        )
        ntk_similarities[i, j] = ntk_similarities[j, i] = ntk_sim['kernel_alignment']

In [ ]:
# Visualize similarity matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

similarity_data = [
    ('Gradient Flow', flow_similarities),
    ('Loss Landscape', landscape_similarities),
    ('Hessian', hessian_similarities),
    ('NTK', ntk_similarities)
]

for idx, (title, sim_matrix) in enumerate(similarity_data):
    ax = axes[idx // 2, idx % 2]
    
    sns.heatmap(sim_matrix, 
                annot=True, 
                fmt='.3f',
                cmap='viridis',
                xticklabels=model_names,
                yticklabels=model_names,
                square=True,
                cbar_kws={'label': 'Similarity'},
                ax=ax)
    
    ax.set_title(f'{title} Similarity')

plt.tight_layout()
plt.show()

## 9. Optimization Dynamics Comparison

In [ ]:
# Track optimization dynamics for a subset of models
selected_models = ['shallow', 'deep']
dynamics = {}

for name in selected_models:
    print(f"\nTracking optimization dynamics for {name} model...")
    
    # Reset model
    model = models[name]
    for layer in model.modules():
        if hasattr(layer, 'reset_parameters'):
            layer.reset_parameters()
    
    # Track dynamics
    dyn = analyzer.track_optimization_dynamics(
        model, loss_fn, train_loader,
        optimizer_class=optim.Adam,
        lr=0.001,
        num_epochs=3
    )
    dynamics[name] = dyn

In [ ]:
# Compare dynamics
dynamics_sim = analyzer.compare_optimization_dynamics(
    dynamics['shallow'], dynamics['deep']
)

print("\nOptimization Dynamics Comparison:")
for key, value in dynamics_sim.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

In [ ]:
# Visualize dynamics comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

metrics = ['loss', 'gradient_norm', 'parameter_change', 'effective_lr']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    for name in selected_models:
        if metric in dynamics[name]:
            values = dynamics[name][metric]
            ax.plot(range(len(values)), values, marker='o', label=name)
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title(f'{metric.replace("_", " ").title()} Evolution')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Combined Similarity Analysis

In [ ]:
# Compute overall similarity by combining all metrics
overall_similarity = np.zeros((n_models, n_models))

# Weight different similarity measures
weights = {
    'flow': 0.2,
    'landscape': 0.3,
    'hessian': 0.3,
    'ntk': 0.2
}

overall_similarity = (
    weights['flow'] * flow_similarities +
    weights['landscape'] * landscape_similarities +
    weights['hessian'] * hessian_similarities +
    weights['ntk'] * ntk_similarities
)

# Visualize overall similarity
plt.figure(figsize=(8, 6))
sns.heatmap(overall_similarity,
            annot=True,
            fmt='.3f',
            cmap='viridis',
            xticklabels=model_names,
            yticklabels=model_names,
            square=True,
            cbar_kws={'label': 'Overall Similarity'})

plt.title('Overall Model Similarity (Weighted Average)')
plt.tight_layout()
plt.show()

In [ ]:
# Perform hierarchical clustering based on similarity
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# Convert similarity to distance
distance_matrix = 1 - overall_similarity
condensed_distances = squareform(distance_matrix)

# Perform hierarchical clustering
linkage_matrix = linkage(condensed_distances, method='average')

# Plot dendrogram
plt.figure(figsize=(10, 6))
dendrogram(linkage_matrix, labels=model_names)
plt.title('Model Clustering Based on Gradient Similarity')
plt.xlabel('Model')
plt.ylabel('Distance')
plt.show()

## Summary

This notebook demonstrated various gradient-based similarity measures:

1. **Gradient Flow Analysis**: Tracked optimization trajectories and compared their properties
2. **Loss Landscape Analysis**: Examined local geometry around model parameters
3. **Hessian Analysis**: Analyzed second-order curvature information
4. **Neural Tangent Kernel**: Compared function space behavior
5. **Optimization Dynamics**: Tracked and compared learning behavior

Key findings:
- Models with similar architectures (e.g., similar depth) show higher gradient flow similarity
- Wider networks tend to have smoother loss landscapes
- NTK similarity captures functional behavior independent of parameterization
- Combined metrics provide a comprehensive view of model similarity

These gradient-based methods complement topological analysis by capturing optimization and functional properties of neural networks.